# Train CLI Smoke Test With Sample `.osz` Files

This notebook is a manual test harness for the refactored CLI.

It does four things:
1. Trains the baseline `taiko_transformer` for 1 epoch from raw sample data.
2. Resumes the baseline checkpoint and continues training to epoch 2.
3. Trains the new `taiko_context_transformer` for 1 epoch from raw sample data.
4. Resumes the long-context checkpoint and continues training to epoch 2.

The cells are intentionally not executed here.


In [ ]:
from pathlib import Path

import torch

from src.model.train_cli import main as train_main

repo_root = Path.cwd()
raw_osz_glob = repo_root / "sample_data" / "raw" / "*.osz"
baseline_data_root = repo_root / "sample_data" / "train_cli_demo"
context_data_root = repo_root / "sample_data" / "train_cli_context_demo"

baseline_training_dir = baseline_data_root / "training"
context_training_dir = context_data_root / "training"

baseline_checkpoint_path = baseline_training_dir / "checkpoints" / "last.ckpt"
context_checkpoint_path = context_training_dir / "checkpoints" / "last.ckpt"

if torch.cuda.is_available():
    best_device = "cuda"
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    best_device = "mps"
else:
    best_device = "cpu"

print(f"repo_root               : {repo_root}")
print(f"raw_osz_glob            : {raw_osz_glob}")
print(f"baseline_data_root      : {baseline_data_root}")
print(f"context_data_root       : {context_data_root}")
print(f"baseline_checkpoint     : {baseline_checkpoint_path}")
print(f"context_checkpoint      : {context_checkpoint_path}")
print(f"best_device             : {best_device}")


## Baseline Model

### Step 1: Prepare intermediates and train the baseline architecture for 1 epoch

This call uses the raw `.osz` files directly. The preprocessing outputs and dataset artifacts are written under `sample_data/train_cli_demo/`.


In [ ]:
baseline_first_run_args = [
    str(raw_osz_glob),
    "--data-root", str(baseline_data_root),
    "--epochs", "1",
    "--batch-size", "4",
    "--lr", "0.001",
    "--device", best_device,
    "--architecture-name", "taiko_transformer",
    "--d-model", "32",
    "--nhead", "4",
    "--num-encoder-layers", "1",
    "--num-decoder-layers", "1",
    "--dim-feedforward", "64",
    "--max-len", "256",
]

print("Running baseline training pass...")
rc = train_main(baseline_first_run_args)
print(f"return code: {rc}")
assert rc == 0, "Baseline training run failed"
assert baseline_checkpoint_path.exists(), "Baseline checkpoint was not written after the first run"


### Step 2: Resume the baseline checkpoint and continue to epoch 2


In [ ]:
baseline_resume_run_args = [
    "--resume-checkpoint", str(baseline_checkpoint_path),
    "--epochs", "2",
    "--batch-size", "4",
    "--device", best_device,
]

print("Running resumed baseline training pass...")
rc = train_main(baseline_resume_run_args)
print(f"return code: {rc}")
assert rc == 0, "Baseline resume training run failed"


## Long-Context Model

### Step 3: Prepare intermediates and train the long-context architecture for 1 epoch

This call uses the raw `.osz` files directly. The preprocessing outputs and dataset artifacts are written under `sample_data/train_cli_context_demo/`.


In [ ]:
context_first_run_args = [
    str(raw_osz_glob),
    "--data-root", str(context_data_root),
    "--epochs", "1",
    "--batch-size", "4",
    "--lr", "0.001",
    "--device", best_device,
    "--architecture-name", "taiko_context_transformer",
    "--d-model", "32",
    "--nhead", "4",
    "--num-encoder-layers", "1",
    "--num-decoder-layers", "1",
    "--dim-feedforward", "64",
    "--max-len", "512",
    "--history-max-tokens", "256",
    "--retrieval-top-k", "2",
    "--retrieval-max-tokens-per-window", "32",
    "--retrieval-exclude-last-n-windows", "1",
    "--use-motif-retrieval",
]

print("Running long-context training pass...")
rc = train_main(context_first_run_args)
print(f"return code: {rc}")
assert rc == 0, "Long-context training run failed"
assert context_checkpoint_path.exists(), "Long-context checkpoint was not written after the first run"


### Step 4: Resume the long-context checkpoint and continue to epoch 2


In [ ]:
context_resume_run_args = [
    "--resume-checkpoint", str(context_checkpoint_path),
    "--epochs", "2",
    "--batch-size", "4",
    "--device", best_device,
]

print("Running resumed long-context training pass...")
rc = train_main(context_resume_run_args)
print(f"return code: {rc}")
assert rc == 0, "Long-context resume training run failed"


## Optional inspection

Use this after the four runs complete to inspect the written artifacts.


In [ ]:
print("baseline splits:", baseline_training_dir / "splits.json")
print("baseline vocab :", baseline_training_dir / "vocab.json")
print("baseline last  :", baseline_checkpoint_path)

print("context splits :", context_training_dir / "splits.json")
print("context vocab  :", context_training_dir / "vocab.json")
print("context last   :", context_checkpoint_path)
print("context arch   : taiko_context_transformer")

for path in [
    baseline_data_root / "unpacked",
    baseline_data_root / "chart_index",
    baseline_data_root / "beat_aligned_dataset",
    baseline_training_dir,
    context_data_root / "unpacked",
    context_data_root / "chart_index",
    context_data_root / "beat_aligned_dataset",
    context_training_dir,
]:
    print(path, "exists=", path.exists())
